# Stage A1 — Data Ingestion & Validation

Dual-fuel PINN pipeline | Sandrine Schueller Mafra | PPGEM – UFPR
Supports dissertation Sections 3.1.1–3.1.2.

Notebook version of `A1_data_ingestion.py` — same logic, `polars` +
`plotly` instead of `pandas` + `matplotlib`, split into cells so each
check's result is visible on its own before moving to the next.

**What this notebook does:**
1. Loads `data/masters_data.xlsx` and renames columns to short,
   code-friendly names.
2. Checks completeness (missing values, duplicate rows).
3. Cross-checks the real min/max/mean/std of every variable against
   **Table 4** (inputs) and **Table 5** (outputs) as currently written
   in the text, and against the *"five to seven levels per variable"*
   claim in Section 3.1.1.2 — flagging mismatches instead of assuming
   the text is right. (This is the one stage where thesis-text values
   are used at all, and only as a comparison target, never as an input
   to any computation.)
4. Infers, directly from the data, which input was being swept at each
   row (one-factor-at-a-time design, Sec. 3.1.1.2).
5. Plots distributions and the inferred OFAT structure.

**Input:** `data/masters_data.xlsx`
**Output:** nothing written to disk by default (inspection notebook);
an optional cell at the end saves the cleaned table + data dictionary
if you want them persisted.


## Setup

In [ ]:
import numpy as np
import polars as pl
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path

print("polars ", pl.__version__)
import plotly
print("plotly ", plotly.__version__)

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "code" else Path.cwd()
RAW_PATH = PROJECT_ROOT / "data" / "masters_data.xlsx"
OUT_DIR = PROJECT_ROOT / "outputs"
RAW_PATH


## Reference values transcribed from the dissertation text

Used **only** to cross-check the raw data below — never to clean,
filter, or transform it. Transcribed directly from Table 4 and Table 5
(Sec. 3.1.2.2) and the baseline/levels description in Sec. 3.1.1.2.
Table 5's own footnote already flags its mean/std as *"estimates"*,
which is part of why this cross-check exists.

In [ ]:
COLUMN_MAP = {
    "SOI [o.CA]": "SOI",
    "Lambda [-]": "lambda",
    "Sub. Rate [%]": "sub_rate",
    "Prail [bar]": "P_rail",
    "HC [g/kW.h]": "HC",
    "NOX [ppm]": "NOx",
    "CO2 [%]": "CO2",
    "SO_H [FSN]": "PM",
    "ETA [%]": "eta",
}
INPUT_COLS = ["SOI", "lambda", "sub_rate", "P_rail"]
OUTPUT_COLS = ["HC", "NOx", "CO2", "PM", "eta"]
ALL_COLS = INPUT_COLS + OUTPUT_COLS

# Table 4, Sec. 3.1.2.2 — input variable ranges
TABLE4_REF = {
    "SOI":      dict(min=2.0,    max=14.0,   mean=7.8,    std=3.5,   unit="degCA bTDC"),
    "lambda":   dict(min=1.4,    max=2.8,    mean=2.05,   std=0.42,  unit="-"),
    "sub_rate": dict(min=0.0,    max=80.0,   mean=44.3,   std=23.1,  unit="%"),
    "P_rail":   dict(min=1300.0, max=1800.0, mean=1490.0, std=165.0, unit="bar"),
}
# Table 5, Sec. 3.1.2.2 — output ranges ("*estimates" per the text's own footnote)
# eta expressed as a fraction here (text states 32-42%) for a like-for-like check.
TABLE5_REF = {
    "HC":  dict(min=2.0,   max=8.0,   mean=4.68,  std=1.24,  unit="g/kWh"),
    "NOx": dict(min=200,   max=1200,  mean=648,   std=267,   unit="ppm"),
    "CO2": dict(min=3.0,   max=8.0,   mean=5.12,  std=1.31,  unit="%"),
    "PM":  dict(min=0.5,   max=15.0,  mean=3.21,  std=3.42,  unit="FSN"),
    "eta": dict(min=0.320, max=0.420, mean=0.371, std=0.0112, unit="fraction (text: %)"),
}
CLAIMED_LEVELS_TEXT = "five to seven"  # Sec. 3.1.1.2 claim


## 1. Load & clean data

Requires the `fastexcel` engine for `polars`: `pip install polars
fastexcel` if the read below errors with a missing-engine message.

In [ ]:
df = pl.read_excel(RAW_PATH)
df = df.rename(COLUMN_MAP).select(ALL_COLS)

print(df.shape)
df.head()


## 2. Completeness — missing values & duplicate rows

In [ ]:
n_missing = sum(df[c].null_count() for c in df.columns)
n_dupes = df.shape[0] - df.unique().shape[0]

print(f"Missing values : {n_missing}  ({'OK' if n_missing == 0 else 'FAIL'})")
print(f"Duplicate rows : {n_dupes}  ({'OK' if n_dupes == 0 else 'FAIL'})")


## 3. Descriptive statistics & distinct-level counts

In [ ]:
desc = df.describe()
desc


In [ ]:
n_levels = pl.DataFrame({
    "variable": ALL_COLS,
    "n_levels_rounded": [df[c].round(1).n_unique() for c in ALL_COLS],
})
n_levels


## 4. Inputs vs. Table 4 (Sec. 3.1.2.2)

`OUT OF RANGE` means the real min/max falls outside what Table 4 claims
— that's Table 4 needing an update, not the data being wrong.

In [ ]:
rows = []
for c, ref in TABLE4_REF.items():
    real_min, real_max = df[c].min(), df[c].max()
    real_mean, real_std = df[c].mean(), df[c].std()
    mismatch = (real_min < ref["min"] - 1e-6) or (real_max > ref["max"] + 1e-6)
    rows.append({
        "variable": c,
        "data_min": real_min, "data_max": real_max,
        "data_mean": round(real_mean, 3), "data_std": round(real_std, 3),
        "table4_min": ref["min"], "table4_max": ref["max"],
        "table4_mean": ref["mean"], "table4_std": ref["std"],
        "status": "OUT OF RANGE" if mismatch else "range OK",
    })
inputs_check = pl.DataFrame(rows)
inputs_check


## 5. Outputs vs. Table 5 (Sec. 3.1.2.2, marked '*estimates' in the text)

In [ ]:
rows = []
for c, ref in TABLE5_REF.items():
    real_min, real_max = df[c].min(), df[c].max()
    real_mean, real_std = df[c].mean(), df[c].std()
    mismatch = (real_min < ref["min"] - 1e-6) or (real_max > ref["max"] + 1e-6)
    rows.append({
        "variable": c,
        "data_min": round(real_min, 3), "data_max": round(real_max, 3),
        "data_mean": round(real_mean, 3), "data_std": round(real_std, 4),
        "table5_min": ref["min"], "table5_max": ref["max"],
        "table5_mean": ref["mean"], "table5_std": ref["std"],
        "status": "OUT OF RANGE" if mismatch else "range OK",
    })
outputs_check = pl.DataFrame(rows)
outputs_check


## 6. OFAT level-count claim vs. Sec. 3.1.1.2

In [ ]:
rows = []
for c in INPUT_COLS:
    n = df[c].round(1).n_unique()
    rows.append({
        "variable": c,
        "n_levels_observed": n,
        "matches_claim": 5 <= n <= 7,
    })
levels_check = pl.DataFrame(rows)
print(f"Claim in the text: '{CLAIMED_LEVELS_TEXT} levels' per input")
levels_check


## 7. Unit check — eta

In [ ]:
eta_min, eta_max = df["eta"].min(), df["eta"].max()
if eta_max <= 1.0:
    print(f"eta is stored as a FRACTION ({eta_min:.3f}-{eta_max:.3f}); "
          f"the text reports it as a PERCENT (e.g. '32-42%'). Multiply by "
          f"100 before quoting eta next to text values, or relabel axes.")
else:
    print(f"eta range is [{eta_min:.2f}, {eta_max:.2f}] -- already percent-scale.")


## 8. Distributions — histogram + boxplot per variable

One variable per iteration, two panels each — run the cell and scroll
through all nine.

In [ ]:
for col in ALL_COLS:
    values = df[col].to_numpy()
    fig = make_subplots(rows=1, cols=2,
                         subplot_titles=[f"{col} - histogram", f"{col} - boxplot"])
    fig.add_trace(go.Histogram(x=values, marker_color="#185FA5", showlegend=False),
                  row=1, col=1)
    fig.add_trace(go.Box(y=values, marker_color="#185FA5", showlegend=False,
                          boxpoints="outliers"),
                  row=1, col=2)
    fig.update_layout(height=320, width=720, title_text=col, showlegend=False)
    fig.show()


## 9. Inferred OFAT block per row

Data-driven, not from the text: for each row, whichever input deviates
most from its own dataset median — normalised by that input's own
range — is taken as the swept variable for that row. A light
neighbour-smoothing pass fixes isolated single-row mismatches (see the
caveat below).

**Caveat:** `lambda` has the narrowest absolute range of the four
inputs, so small measurement jitter in `lambda` during a *different*
variable's sweep can look like a large deviation once normalised.
Treat this label as a diagnostic aid, not an authoritative experimental
log — cross-check against a lab notebook / run order if you have one.

In [ ]:
medians = {c: df[c].median() for c in INPUT_COLS}
ranges = {c: (df[c].max() - df[c].min()) for c in INPUT_COLS}

deviation = np.column_stack([
    np.abs(df[c].to_numpy() - medians[c]) / ranges[c] for c in INPUT_COLS
])
raw_labels = np.array(INPUT_COLS)[deviation.argmax(axis=1)]


def smooth_isolated_labels(labels, passes=2):
    out = list(labels)
    for _ in range(passes):
        changed = False
        for i in range(1, len(out) - 1):
            if out[i] != out[i - 1] and out[i - 1] == out[i + 1]:
                out[i] = out[i - 1]
                changed = True
        if not changed:
            break
    return np.array(out)


block_labels = smooth_isolated_labels(raw_labels)
df = df.with_columns(pl.Series("ofat_block", block_labels))

from collections import Counter
counts = Counter(block_labels)
block_counts = pl.DataFrame({
    "ofat_block": list(counts.keys()),
    "count": list(counts.values()),
}).sort("count", descending=True)
block_counts


## 10. OFAT block scatter — inputs vs. row index, coloured by inferred block

In [ ]:
colors = {"SOI": "#185FA5", "lambda": "#993C1D", "sub_rate": "#534AB7", "P_rail": "#3B6D11"}
row_ids = np.arange(df.shape[0])

fig = make_subplots(rows=2, cols=2, subplot_titles=INPUT_COLS)
for i, col in enumerate(INPUT_COLS):
    r, c = divmod(i, 2)
    col_values = df[col].to_numpy()
    for block, color in colors.items():
        mask = block_labels == block
        fig.add_trace(
            go.Scatter(x=row_ids[mask], y=col_values[mask], mode="markers",
                       marker=dict(color=color, size=8), name=block,
                       showlegend=(i == 0)),
            row=r + 1, col=c + 1,
        )
fig.update_layout(height=650, width=800,
                   title_text="Inputs vs. row index, coloured by inferred OFAT block")
fig.show()


## 11. Data dictionary

In [ ]:
data_dictionary = pl.DataFrame({
    "column":  ["SOI", "lambda", "sub_rate", "P_rail", "HC", "NOx", "CO2", "PM", "eta"],
    "role":    ["input", "input", "input", "input",
                "output", "output", "output", "output", "output"],
    "description": [
        "Start of injection",
        "Excess air ratio (relative air-fuel ratio)",
        "Diesel substitution rate by natural gas",
        "Injection (common-rail) pressure",
        "Unburned hydrocarbon emissions",
        "Nitrogen oxide emissions",
        "Carbon dioxide concentration in exhaust",
        "Particulate matter emissions (text calls this PM)",
        "Brake thermal efficiency (fraction 0-1; text reports as %)",
    ],
    "unit": ["degCA bTDC", "-", "%", "bar", "g/kWh", "ppm", "%", "FSN", "fraction"],
    "raw_column_name": ["SOI [o.CA]", "Lambda [-]", "Sub. Rate [%]", "Prail [bar]",
                         "HC [g/kW.h]", "NOX [ppm]", "CO2 [%]", "SO_H [FSN]", "ETA [%]"],
    "source": ["Lima Nogueira (2019), Table 3/4/5"] * 9,
})
data_dictionary


## Optional — persist outputs

Only run this if you want the cleaned table and data dictionary saved
to `outputs/`; nothing downstream in this notebook depends on it.

In [ ]:
OUT_DIR.mkdir(parents=True, exist_ok=True)
df.write_csv(OUT_DIR / "A1_clean_data_with_blocks.csv")
data_dictionary.write_csv(OUT_DIR / "A1_data_dictionary.csv")
print(f"Saved to {OUT_DIR}")


## Next

Continue with **A2** (`A2_exploratory_analysis.ipynb`) — correlation,
VIF, KDE — which reads `masters_data.xlsx` independently, so it doesn't
need anything saved from this notebook to run.